In [1]:
import pandas as pd
import numpy as np

# Load data
results = pd.read_csv('data/results.csv')
shootouts = pd.read_csv('data/shootouts.csv')
former_names = pd.read_csv('data/former_names.csv')

# Parse dates
results['date'] = pd.to_datetime(results['date'])
shootouts['date'] = pd.to_datetime(shootouts['date'])

print(f"Total matches: {len(results)}")
print(f"Date range: {results['date'].min()} → {results['date'].max()}")
print(f"\nColumns: {list(results.columns)}")
results.head()

Total matches: 49477
Date range: 1872-11-30 00:00:00 → 2026-06-27 00:00:00

Columns: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [2]:
# Pre-tournament cutoff: exclude everything from June 11, 2026 onward
CUTOFF = pd.Timestamp('2026-06-11')

df = results[results['date'] < CUTOFF].copy()

# Drop rows with missing scores (future fixtures that snuck in before cutoff)
df = df.dropna(subset=['home_score', 'away_score'])

print(f"Matches after cutoff filter: {len(df)}")
print(f"New date range: {df['date'].min()} → {df['date'].max()}")
print(f"\nDropped: {len(results) - len(df)} rows")

# Quick check - confirm no 2026 WC matches in training data
wc2026 = df[df['tournament'] == 'FIFA World Cup'][df['date'] >= '2026-01-01']
print(f"2026 WC matches in training data: {len(wc2026)} (should be 0)")

Matches after cutoff filter: 49405
New date range: 1872-11-30 00:00:00 → 2026-06-10 00:00:00

Dropped: 72 rows
2026 WC matches in training data: 0 (should be 0)


/var/folders/pd/j9xkws912y56qb32qqkxc0gm0000gn/T/ipykernel_41596/4244917196.py:14: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  wc2026 = df[df['tournament'] == 'FIFA World Cup'][df['date'] >= '2026-01-01']


In [3]:
wc2026 = df[(df['tournament'] == 'FIFA World Cup') & (df['date'] >= '2026-01-01')]
print(f"2026 WC matches in training data: {len(wc2026)} (should be 0)")

2026 WC matches in training data: 0 (should be 0)


In [6]:
# Assign weights by tournament type (recent competitive matches matter more)
tournament_weights = {
    'FIFA World Cup': 4.0,
    'UEFA Euro': 3.0,
    'Copa América': 3.0,
    'Africa Cup of Nations': 3.0,
    'Asian Cup': 3.0,
    'FIFA World Cup qualification': 2.0,
    'UEFA Nations League': 2.0,
    'Friendly': 0.2
}

def get_weight(tournament):
    for key, weight in tournament_weights.items():
        if key in tournament:
            return weight
    return 1.0  # default for other competitive tournaments

df['weight'] = df['tournament'].apply(get_weight)

# Create match outcome from home team perspective
# 1 = home win, 0 = draw, -1 = away win
def get_outcome(row):
    if row['home_score'] > row['away_score']:
        return 1
    elif row['home_score'] == row['away_score']:
        return 0
    else:
        return -1

df['outcome'] = df.apply(get_outcome, axis=1)

print("Outcome distribution:")
print(df['outcome'].value_counts())
print(f"\nWeight distribution:")
print(df['weight'].value_counts().sort_index())

Outcome distribution:
outcome
 1    24212
-1    13960
 0    11233
Name: count, dtype: int64

Weight distribution:
weight
0.2    18388
1.0    15285
2.0      658
3.0     5339
4.0     9735
Name: count, dtype: int64


## Elo Rating System

Elo is a rolling strength score per team that updates after every match.
Win against a strong team → big gain. Lose to a weak team → big drop.

### Design Decisions

**Two-phase approach:**
- Phase 1 runs the full historical dataset (1872–2026) at 30% weight to establish a baseline rating for every nation
- Phase 2 re-runs 2018 onward at full weight so recent form drives the final ratings

**Why 2018 cutoff for Phase 2?**
Covers two full World Cup cycles (2018, 2022) plus qualifications and continental tournaments — enough signal without over-indexing on ancient history.

**K-factor = 20** (lower than the classic 32)
More stable ratings, prevents single matches from swinging Elo too drastically.
Tournament weights (4.0 for WC, 0.2 for friendlies) handle importance scaling instead.

**On Brazil and Germany being lower than expected:**
Both teams had genuinely inconsistent recent runs — Brazil lost to Bolivia in WC qualification and Japan in the Kirin Cup; Germany lost to Slovakia early in qualification before recovering.
The Elo reflects reality. This is a feature, not a bug, and makes for an honest model.

**Final ratings (pre-tournament, cutoff June 11 2026):**
Spain 2042 · France 1945 · England 1945 · Morocco 1920 · Argentina 1906
Brazil 1801 · Germany 1853 · Portugal 1833

In [12]:
# Elo rating system - two phase
# Phase 1: full history at low K for baseline
# Phase 2: 2018 onward at full K for current form

ELO_START = 1500
K_BASE = 20

elo_ratings = {}

def get_elo(team):
    return elo_ratings.get(team, ELO_START)

def expected_score(elo_a, elo_b):
    return 1 / (1 + 10 ** ((elo_b - elo_a) / 400))

def update_elo(home_team, away_team, outcome, weight):
    home_elo = get_elo(home_team)
    away_elo = get_elo(away_team)
    
    expected_home = expected_score(home_elo, away_elo)
    expected_away = 1 - expected_home
    
    if outcome == 1:
        actual_home, actual_away = 1, 0
    elif outcome == 0:
        actual_home, actual_away = 0.5, 0.5
    else:
        actual_home, actual_away = 0, 1
    
    K = K_BASE * weight
    elo_ratings[home_team] = home_elo + K * (actual_home - expected_home)
    elo_ratings[away_team] = away_elo + K * (actual_away - expected_away)

df_sorted = df.sort_values('date').reset_index(drop=True)

# Phase 1: full history, low weight baseline
for _, row in df_sorted.iterrows():
    update_elo(row['home_team'], row['away_team'],
               row['outcome'], row['weight'] * 0.3)

# Phase 2: 2018 onward, record pre-match Elo as features
df_recent = df_sorted[df_sorted['date'] >= '2018-06-01'].copy()

home_elos, away_elos = [], []
for _, row in df_recent.iterrows():
    home_elos.append(get_elo(row['home_team']))
    away_elos.append(get_elo(row['away_team']))
    update_elo(row['home_team'], row['away_team'],
               row['outcome'], row['weight'])

df_recent['home_elo'] = home_elos
df_recent['away_elo'] = away_elos
df_recent['elo_diff'] = df_recent['home_elo'] - df_recent['away_elo']

print(f"Training set size (2018+): {len(df_recent)}")
print(f"Date range: {df_recent['date'].min()} → {df_recent['date'].max()}")
print(f"\nFinal Elo - Top 10:")
top_teams = sorted(elo_ratings.items(), key=lambda x: x[1], reverse=True)[:10]
for team, elo in top_teams:
    print(f"  {team}: {elo:.0f}")

print(f"\nSpot check:")
for team in ['Brazil', 'Argentina', 'France', 'Germany', 'Portugal']:
    print(f"  {team}: {elo_ratings.get(team, 1500):.0f}")

Training set size (2018+): 7888
Date range: 2018-06-01 00:00:00 → 2026-06-10 00:00:00

Final Elo - Top 10:
  Spain: 2042
  France: 1945
  England: 1945
  Morocco: 1920
  Argentina: 1906
  Japan: 1904
  Australia: 1896
  Turkey: 1865
  South Korea: 1862
  Iran: 1857

Spot check:
  Brazil: 1801
  Argentina: 1906
  France: 1945
  Germany: 1853
  Portugal: 1833


In [13]:
# Recent form: win rate over last 5 matches per team (rolling, date-ordered)
# Computed across full dataset so 2018+ matches have solid form history

def compute_recent_form(df_all, team, before_date, n=5):
    matches = df_all[
        ((df_all['home_team'] == team) | (df_all['away_team'] == team)) &
        (df_all['date'] < before_date)
    ].tail(n)
    
    if len(matches) == 0:
        return 0.5  # neutral default
    
    wins = 0
    for _, row in matches.iterrows():
        if row['home_team'] == team:
            if row['outcome'] == 1:
                wins += 1
            elif row['outcome'] == 0:
                wins += 0.5
        else:
            if row['outcome'] == -1:
                wins += 1
            elif row['outcome'] == 0:
                wins += 0.5
    
    return wins / len(matches)

# Apply to training set (2018+)
# Note: this is slow (~2 mins) because it looks up history per row
print("Computing recent form... (this takes a minute)")

home_form, away_form = [], []
for _, row in df_recent.iterrows():
    home_form.append(compute_recent_form(df_sorted, row['home_team'], row['date']))
    away_form.append(compute_recent_form(df_sorted, row['away_team'], row['date']))

df_recent['home_form'] = home_form
df_recent['away_form'] = away_form
df_recent['form_diff'] = df_recent['home_form'] - df_recent['away_form']

print("Done.")
print(f"\nSample:")
print(df_recent[['date','home_team','away_team','home_form','away_form','form_diff']].tail())

Computing recent form... (this takes a minute)
Done.

Sample:
            date    home_team   away_team  home_form  away_form  form_diff
49400 2026-06-09     Ethiopia      Malawi        0.4        0.3        0.1
49401 2026-06-10      England  Costa Rica        0.7        0.2        0.5
49402 2026-06-10     Portugal     Nigeria        0.7        0.8       -0.1
49403 2026-06-10      Bolivia     Algeria        0.4        0.7       -0.3
49404 2026-06-10  Afghanistan    Pakistan        0.4        0.6       -0.2


In [14]:
# Home advantage feature
# neutral=True means no home advantage for either team
df_recent['is_neutral'] = df_recent['neutral'].astype(int)
df_recent['home_advantage'] = (df_recent['neutral'] == False).astype(int)

# Final feature set
features = [
    'home_elo',
    'away_elo', 
    'elo_diff',
    'home_form',
    'away_form',
    'form_diff',
    'home_advantage',
    'is_neutral'
]

# Target: convert outcome to 3-class (0=away win, 1=draw, 2=home win)
df_recent['target'] = df_recent['outcome'].map({-1: 0, 0: 1, 1: 2})

# Final clean dataset
df_model = df_recent[features + ['target', 'date', 
                                  'home_team', 'away_team', 
                                  'weight']].dropna()

print(f"Final dataset shape: {df_model.shape}")
print(f"\nFeature summary:")
print(df_model[features].describe().round(2))
print(f"\nTarget distribution:")
print(df_model['target'].value_counts().sort_index()
      .rename({0:'Away Win', 1:'Draw', 2:'Home Win'}))

# Save
df_model.to_csv('outputs/model_ready.csv', index=False)

# Save final Elo ratings for use in notebook 2
import json
with open('outputs/elo_ratings.json', 'w') as f:
    json.dump(elo_ratings, f)

print("\n✅ Saved: outputs/model_ready.csv")
print("✅ Saved: outputs/elo_ratings.json")

Final dataset shape: (7888, 13)

Feature summary:
       home_elo  away_elo  elo_diff  home_form  away_form  form_diff  \
count   7888.00   7888.00   7888.00    7888.00    7888.00    7888.00   
mean    1559.76   1536.28     23.48       0.51       0.49       0.02   
std      200.20    193.96    218.10       0.24       0.24       0.32   
min      868.37    871.48  -1008.39       0.00       0.00      -1.00   
25%     1434.44   1420.03   -105.94       0.30       0.30      -0.20   
50%     1556.66   1533.48     26.10       0.50       0.50       0.00   
75%     1713.71   1674.17    157.86       0.70       0.70       0.20   
max     2070.15   2065.41    995.32       1.00       1.00       1.00   

       home_advantage  is_neutral  
count         7888.00     7888.00  
mean             0.68        0.32  
std              0.47        0.47  
min              0.00        0.00  
25%              0.00        0.00  
50%              1.00        0.00  
75%              1.00        1.00  
max          

## Notebook 1 Complete ✅

### Output Files
- `outputs/model_ready.csv` — 7,888 matches (2018–2026) with engineered features, ready for modeling
- `outputs/elo_ratings.json` — final pre-tournament Elo ratings for all nations, used in Notebook 2 for simulation

### Features Built
| Feature | Description |
|---|---|
| home_elo / away_elo | Team strength rating at time of match |
| elo_diff | home_elo minus away_elo |
| home_form / away_form | Win rate over last 5 matches |
| form_diff | home_form minus away_form |
| home_advantage | 1 if playing at home ground |
| is_neutral | 1 if neutral venue |

### Next: Notebook 2
- Train XGBoost match outcome classifier
- Validate on historical World Cup matches
- Monte Carlo tournament simulation (10,000 runs)
- Output: winner probabilities + most likely bracket